In [2]:
import pandas as pd

In [3]:
path = "D:/Ai-adoption-intel/data/raw/National.xlsx"
pd.ExcelFile(path).sheet_names

['Response Estimates',
 'Response Standard Errors',
 'Index Estimates',
 'Index Standard Errors',
 'Collection and Reference Dates',
 'Data Dictionary']

In [4]:
df = pd.read_excel(path,sheet_name="Response Estimates")
df.head()

,Question ID,Question,Answer ID,Answer,202620,202619,202618,202617,202616,202615,...,202402,202401,202326,202325,202324,202323,202322,202321,202320,202319
0,3,"Overall, how would you describe this business'...",1.0,Excellent,14.7%,14.7%,14.6%,14.7%,14.8%,12.1%,...,10.7%,11.8%,12.0%,12.0%,13.3%,13.4%,13.4%,13.8%,13.8%,13.3%
1,3,"Overall, how would you describe this business'...",2.0,Above average,20.5%,21.2%,21.3%,21.1%,20.8%,19.8%,...,19.9%,20.4%,20.8%,21.0%,22.1%,22.0%,21.8%,22.1%,21.9%,22.0%
2,3,"Overall, how would you describe this business'...",3.0,Average,45.6%,45.5%,45.3%,46.4%,46.0%,47.8%,...,47.5%,47.0%,46.1%,45.9%,45.2%,45.2%,45.1%,44.8%,45.3%,45.7%
3,3,"Overall, how would you describe this business'...",4.0,Below average,15.2%,14.7%,14.8%,14.0%,14.5%,15.9%,...,16.1%,15.1%,15.1%,15.3%,14.5%,14.4%,14.7%,14.2%,14.2%,14.5%
4,3,"Overall, how would you describe this business'...",5.0,Poor,4.0%,4.0%,3.9%,3.9%,3.9%,4.4%,...,5.7%,5.8%,6.0%,5.8%,5.0%,5.0%,5.0%,5.1%,4.8%,4.5%


In [5]:
Shape = df.shape
print(f"The Shape of Sheet Response Estimators: {Shape}")
print(f"The number of rows: {Shape[0]}")
print(f"The number of columns: {Shape[1]}")

The Shape of Sheet Response Estimators: (85, 84)
The number of rows: 85
The number of columns: 84


Listing every unique question in response estimators.

In [6]:
df[["Question ID","Question"]].drop_duplicates()

,Question ID,Question
0,3,"Overall, how would you describe this business'..."
5,4,"In the last two weeks, how did this business's..."
8,5,"In the last two weeks, how did this business's..."
11,6,"In the last two weeks, how did the total numbe..."
14,7,"In the last two weeks, did this business use A..."
17,8,"In the last two weeks, how did the time it tak..."
21,9,How would you describe this business's current...
25,10,"In the last two weeks, how did demand for this..."
28,11,"In the last two weeks, how did the prices this..."
31,12,"In the last two weeks, how did the prices this..."


The smpdt is a year+two-week-period . 202620 is 2026 year and 20th fortnight of 2026. Census run the survey every two weeks, so here each column is one survey round.

The quetion 7 and Quetion 24 are AI related ones.

Quetion7: In the last two weeks, did this business use Artificial Intelligence (AI) in any of its business functions? (Examples of AI: machine learning, natural language processing, virtual agents, voice recognition, etc.)

Quetion24: During the next six months, do you think this business will be using Artificial Intelligence (AI) in any of its business functions? (Examples of AI: machine learning, natural language processing, virtual agents, voice recognition, etc.)

For Quetion 7 and 24 we have data from 202620 to 202524 only where this file has data AI used has taken from late 2025.

In [7]:
df[df["Question ID"]==7]

,Question ID,Question,Answer ID,Answer,202620,202619,202618,202617,202616,202615,...,202402,202401,202326,202325,202324,202323,202322,202321,202320,202319
14,7,"In the last two weeks, did this business use A...",1.0,Yes,23.8%,23.8%,23.2%,22.4%,21.8%,21.5%,...,.,.,.,.,.,.,.,.,.,.
15,7,"In the last two weeks, did this business use A...",2.0,No,66.4%,66.5%,67.2%,68.2%,68.6%,68.1%,...,.,.,.,.,.,.,.,.,.,.
16,7,"In the last two weeks, did this business use A...",3.0,Do not know,9.8%,9.7%,9.6%,9.4%,9.6%,10.3%,...,.,.,.,.,.,.,.,.,.,.


In [8]:
df.tail(3)

,Question ID,Question,Answer ID,Answer,202620,202619,202618,202617,202616,202615,...,202402,202401,202326,202325,202324,202323,202322,202321,202320,202319
82,24,"During the next six months, do you think this ...",3.0,Do not know,25.9%,26.1%,25.8%,26.3%,25.9%,26.2%,...,.,.,.,.,.,.,.,.,.,.
83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,"Source: U.S. Census Bureau, Business Trends an...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The . means not available, The question was not asked at that time.

The S means Surpressed. Census hides a number when few business answers like fewer responses.

In [9]:
print(f"The Starnge values in this sheet with S: {(df=='S').sum().sum()}")
print(f"The Starnge values in this sheet with .: {(df=='.').sum().sum()}")


The Starnge values in this sheet with S: 18
The Starnge values in this sheet with .: 903


In [10]:
qid = pd.to_numeric(df["Question ID"],errors = "coerce")
df_cleaned = df[qid.notna()]

In [11]:
print(f"The Shape of the this sheet after removing the footnote and an empty rows: {df_cleaned.shape}")

The Shape of the this sheet after removing the footnote and an empty rows: (83, 84)


If we use iloc to remove the footnote, it works but if they update any data in futher we may not get the data

Chaning the wide range of columns in to rows as long.

In [12]:
date_cols = [c for c in df_cleaned.columns if str(c).isdigit()]
print(len(date_cols))
print(date_cols[:3])

80
['202620', '202619', '202618']


In [15]:
new_df = df_cleaned.melt(
    id_vars = ["Question ID","Question","Answer ID","Answer"],
    value_vars = date_cols,
    var_name = "cycle",
    value_name = "value",  
)
print(f"The Shape of the new dataset after changing from wide to long data: {new_df.shape}")
new_df.head()

The Shape of the new dataset after changing from wide to long data: (6640, 6)


,Question ID,Question,Answer ID,Answer,cycle,value
0,3,"Overall, how would you describe this business'...",1.0,Excellent,202620,14.7%
1,3,"Overall, how would you describe this business'...",2.0,Above average,202620,20.5%
2,3,"Overall, how would you describe this business'...",3.0,Average,202620,45.6%
3,3,"Overall, how would you describe this business'...",4.0,Below average,202620,15.2%
4,3,"Overall, how would you describe this business'...",5.0,Poor,202620,4.0%


In [16]:
new_df["status"] = "ok"
new_df.loc[new_df["value"]=="S","status"] = "suppresed"
new_df.loc[new_df["value"]==".","status"] = "not collected"
new_df["status"].value_counts()

status
ok               5719
not collected     903
suppresed          18
Name: count, dtype: int64

In [17]:
new_df["percentage"] = pd.to_numeric(
    new_df["value"].astype(str).str.rstrip("%"),
    errors = "coerce"
)
new_df[["value","status","percentage"]].sample(5)

,value,status,percentage
5296,2.4%,ok,2.4
571,64.7%,ok,64.7
3447,2.4%,ok,2.4
5915,27.5%,ok,27.5
5418,12.4%,ok,12.4


In [19]:
ai_used = new_df[(new_df["Question ID"] == 7) & (new_df["Answer"] == "Yes") & (new_df["status"] == "ok") ]
ai_used[["cycle","percentage"]]

,cycle,percentage
14,202620,23.8
97,202619,23.8
180,202618,23.2
263,202617,22.4
346,202616,21.8
429,202615,21.5
512,202614,21.7
595,202613,20.6
678,202612,20.6
761,202611,20.6
